# 01 — Exploratory Data Analysis
CS2 Match Outcome Predictor & Seeding Engine

Goals:
1. Load the dataset and confirm its shape
2. Verify which columns are STATIC vs DYNAMIC (per-team check)
3. Detect and quantify the bo1/bo3 mislabeling issue
4. Check other data quality issues (event_type casing, empty columns)

In [ ]:
import os

def find_repo_root(marker='requirements.txt'):
    """Walks up directories until it finds the repo root (marked by requirements.txt)."""
    path = os.getcwd()
    while True:
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root (looking for '{marker}')")
        path = parent

repo_root = find_repo_root()
os.chdir(repo_root)
print("Working directory set to:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_csv('data/cs2_newestcombinedmatches.csv')
df['date'] = pd.to_datetime(df['date'])

print("Shape:", df.shape)
df.head(3)

Shape: (7033, 140)


,match_id,hltv_match_id,date,tournament,winner,season,score_team1,score_team2,winner_map,loser_map,decider_map,winner_head2head_freq,loser_head2head_freq,winner_head2head_percentage,loser_head2head_percentage,winner_past3,loser_past3,winner_mirage,loser_mirage,winner_inferno,loser_inferno,winner_nuke,loser_nuke,winner_dust2,loser_dust2,winner_overpass,loser_overpass,winner_train,loser_train,winner_ancient,loser_ancient,winner_vertigo,loser_vertigo,winner_anubis,loser_anubis,match_type,event_type,scraped_date,hltv_url,match_number,team1_name,team1_avg_DPR,team1_avg_KAST,team1_avg_ADR,team1_avg_KPR,team1_avg_RATING,team1_player_1_name,team1_player_1_DPR,team1_player_1_KAST,team1_player_1_ADR,team1_player_1_KPR,team1_player_1_RATING,team1_player_2_name,team1_player_2_DPR,team1_player_2_KAST,team1_player_2_ADR,team1_player_2_KPR,team1_player_2_RATING,team1_player_3_name,team1_player_3_DPR,team1_player_3_KAST,team1_player_3_ADR,team1_player_3_KPR,team1_player_3_RATING,team1_player_4_name,team1_player_4_DPR,team1_player_4_KAST,team1_player_4_ADR,team1_player_4_KPR,team1_player_4_RATING,team1_player_5_name,team1_player_5_DPR,team1_player_5_KAST,team1_player_5_ADR,team1_player_5_KPR,team1_player_5_RATING,team2_name,team2_avg_DPR,team2_avg_KAST,team2_avg_ADR,team2_avg_KPR,team2_avg_RATING,team2_player_1_name,team2_player_1_DPR,team2_player_1_KAST,team2_player_1_ADR,team2_player_1_KPR,team2_player_1_RATING,team2_player_2_name,team2_player_2_DPR,team2_player_2_KAST,team2_player_2_ADR,team2_player_2_KPR,team2_player_2_RATING,team2_player_3_name,team2_player_3_DPR,team2_player_3_KAST,team2_player_3_ADR,team2_player_3_KPR,team2_player_3_RATING,team2_player_4_name,team2_player_4_DPR,team2_player_4_KAST,team2_player_4_ADR,team2_player_4_KPR,team2_player_4_RATING,team2_player_5_name,team2_player_5_DPR,team2_player_5_KAST,team2_player_5_ADR,team2_player_5_KPR,team2_player_5_RATING,rating_diff,adr_diff,kast_diff,kpr_diff,dpr_diff,team1_wins,team2_wins,team1_losses,team2_losses,team1_totalwinrate,team2_totalwinrate,team1_totallossrate,team2_totallossrate,team1_rating_std,team2_rating_std,consistency_advantage,team1_top_player,team2_top_player,star_player_advantage,team1_weakest_player,team2_weakest_player,weakest_link_advantage,team1_online_winrate,team2_online_winrate,team1_lan_winrate,team2_lan_winrate,team1_overall_winrate,team2_overall_winrate
0,hltv_match_2371997,2371997,2024-05-15 06:00:00+00:00,BetBoom Dacha Belgrade 2024,team1,8,2,0,Ancient,Dust2,Anubis,0.0,0.0,50.00,50.00,71.43,80.00,58.4,60.2,53.3,53.7,58.8,61.6,67.7,47.8,57.3,55.0,59.8,56.8,69.4,53.1,54.5,44.4,67.1,51.9,bo3,LAN,2025-10-31T09:43:19.255392Z,https://www.hltv.org/matches/2371997/-,NaN,Spirit,0.64,73.36,78.98,0.74,1.14,chopper,0.66,70.2,70.2,0.63,0.94,sh1ro,0.54,76.3,78.4,0.77,1.26,tn1r,0.66,72.7,78.1,0.72,1.12,donk,0.67,74.9,90.6,0.86,1.31,zweih,0.68,72.7,77.6,0.70,1.09,Aurora,0.66,71.34,77.60,0.71,1.07,maj3r,0.66,69.6,70.7,0.62,0.93,xantares,0.67,72.6,88.8,0.8,1.16,woxic,0.61,71.9,75.0,0.73,1.11,wicadia,0.70,70.7,77.6,0.72,1.09,jottaaa,0.68,71.9,75.9,0.70,1.08,0.07,1.38,2.02,0.03,-0.02,29,14,7,21,0.805556,0.400000,0.194444,0.600000,0.146731,0.086197,-0.060534,1.31,1.16,0.15,0.94,0.93,0.01,0.000000,0.442308,0.460317,0.379310,0.5,0.5
1,hltv_match_2372188,2372188,2024-05-15 08:00:00+00:00,RES Regional Series 4 Europe,team2,8,1,2,Nuke,Anubis,Inferno,1.0,4.0,20.00,80.00,55.81,64.29,44.6,62.0,48.1,69.5,57.0,53.4,54.3,59.5,43.1,43.8,51.5,42.3,0.0,42.2,41.1,60.7,62.2,60.0,bo3,Online,2025-10-31T09:42:29.755424Z,https://www.hltv.org/matches/2372188/-,NaN,3DMAX,0.66,70.82,73.98,0.67,1.05,bodyy,0.68,70.1,76.4,0.67,0.98,maka,0.63,71.4,74.4,0.71,1.12,lucky,0.64,72.1,76.5,0.68,1.09,ex3rcice,0.65,70.7,71.1,0.66,1.03,graviti,0.71,69.8,71.5,0.64,1.01,Zero Tenacity,0.67,69.98,74.24,0.67,1.02,avn,0.67,69.9,67.4,0.62,0.91,nemanha,0.71,69.4,83.5,0.7,1.00,maden,0.70,68.9,78.3,0.71,1.07,cjoffo,0.64,71.3,70.0,0.65,1.03,brutmonster,0.64,70.4,72.0,0.69,1.09,0.03,-0.26,0.84,0.00,-0.01,34,19,24,25,0.

## 1. Basic dataset overview

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Date range:", df['date'].min(), "-", df['date'].max())
print("Unique teams:", pd.concat([df['team1_name'], df['team2_name']]).nunique())
print("Unique tournaments:", df['tournament'].nunique())

Rows: 7033
Columns: 140
Date range: 2024-05-15 06:00:00+00:00 - 2025-10-16 13:00:00+00:00
Unique teams: 331
Unique tournaments: 648


## Full null-value audit

Checking every column with missing values, including count and
percentage, to catch anything beyond the known winner_map/loser_map
and match_number issues.

In [ ]:
pd.set_option('display.max_rows', None)

nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)

null_summary = pd.DataFrame({
    'null_count': nulls,
    'null_pct': (nulls / len(df) * 100).round(2)
})

print(f"Total columns with nulls: {len(nulls)} / {len(df.columns)}")
null_summary

Total columns with nulls: 71 / 140


,null_count,null_pct
match_number,7033,100.00
loser_map,1010,14.36
winner_map,1010,14.36
team2_player_5_RATING,910,12.94
team2_player_5_KPR,910,12.94
team2_player_5_ADR,910,12.94
team2_player_5_KAST,910,12.94
team2_player_5_DPR,910,12.94
team1_player_5_RATING,854,12.14
team1_player_5_KAST,854,12.14


## 2. Static vs Dynamic column verification

We check whether team/player-level average stats change across matches
for the same team, or are frozen career averages.

In [ ]:
def check_dynamic(df, column_pair, team_name, name_col='team1_name'):
    """
    column_pair: (team1_col, team2_col) e.g. ('team1_avg_RATING', 'team2_avg_RATING')
    Returns the number of unique values this team has across all its matches.
    """
    t1_col, t2_col = column_pair
    vals = pd.concat([
        df.loc[df['team1_name'] == team_name, t1_col],
        df.loc[df['team2_name'] == team_name, t2_col],
    ])
    return vals.nunique(), len(vals)

# Example: check a few known columns for one team
test_team = 'Spirit'
for pair in [
    ('team1_avg_RATING', 'team2_avg_RATING'),
    ('team1_wins', 'team2_wins'),
    ('team1_totalwinrate', 'team2_totalwinrate'),
    ('team1_lan_winrate', 'team2_lan_winrate'),
]:
    uniq, total = check_dynamic(df, pair, test_team)
    status = "DYNAMIC" if uniq > 1 else "STATIC"
    print(f"{pair[0]:30s} -> unique={uniq:4d} / total={total:4d}  [{status}]")

team1_avg_RATING               -> unique=   1 / total= 104  [STATIC]
team1_wins                     -> unique=  46 / total= 104  [DYNAMIC]
team1_totalwinrate             -> unique=  72 / total= 104  [DYNAMIC]
team1_lan_winrate              -> unique=  86 / total= 104  [DYNAMIC]


## 3. Full static/dynamic audit across all candidate columns

Run the same check across every relevant column pair, for a sample of
teams, and summarize which columns are consistently static.

In [ ]:
column_pairs = {
    'avg_RATING': ('team1_avg_RATING', 'team2_avg_RATING'),
    'avg_DPR': ('team1_avg_DPR', 'team2_avg_DPR'),
    'avg_ADR': ('team1_avg_ADR', 'team2_avg_ADR'),
    'rating_std': ('team1_rating_std', 'team2_rating_std'),
    'top_player': ('team1_top_player', 'team2_top_player'),
    'weakest_player': ('team1_weakest_player', 'team2_weakest_player'),
    'wins': ('team1_wins', 'team2_wins'),
    'totalwinrate': ('team1_totalwinrate', 'team2_totalwinrate'),
    'lan_winrate': ('team1_lan_winrate', 'team2_lan_winrate'),
    'online_winrate': ('team1_online_winrate', 'team2_online_winrate'),
}

sample_teams = ['Spirit', 'NAVI', 'G2', 'Vitality']

results = []
for label, pair in column_pairs.items():
    for team in sample_teams:
        uniq, total = check_dynamic(df, pair, team)
        results.append({'column': label, 'team': team, 'unique_values': uniq, 'n_matches': total})

audit_df = pd.DataFrame(results)
summary = audit_df.groupby('column')['unique_values'].max().sort_values()
print(summary)

column
avg_DPR            1
avg_RATING         1
top_player         1
rating_std         1
weakest_player     1
avg_ADR            2
online_winrate     3
wins              53
totalwinrate      72
lan_winrate       91
Name: unique_values, dtype: int64


## 4. bo1/bo3 mislabeling issue

Rows with empty `winner_map` are checked against `score_team1`/`score_team2`
to distinguish genuine bo1 matches from correctly labeled bo3 matches.

In [ ]:
empty_map = df[df['winner_map'].isna() & df['loser_map'].isna()]
print("Rows with empty winner_map & loser_map:", len(empty_map))

# Case A: round-scale scores (>2) -> true bo1, score needs normalizing to 1-0/0-1
condition_a = empty_map['score_team1'] > 2
condition_a |= empty_map['score_team2'] > 2
case_a = empty_map[condition_a]
print("Case A (score > 2, true bo1, match_type + score fix):", len(case_a))

# Case C: score already <=2, but decider_map filled -> also true bo1
remaining = empty_map[~condition_a]
case_c = remaining[remaining['decider_map'].notna()]
print("Case C (score <=2, decider_map filled, true bo1):", len(case_c))

# Forfeit/unknown: score <=2 AND decider_map also empty
case_forfeit = remaining[remaining['decider_map'].isna()]
print("Forfeit/unknown (score <=2, no map info at all):", len(case_forfeit))

Rows with empty winner_map & loser_map: 1010
Case A (score > 2, true bo1, match_type + score fix): 965
Case C (score <=2, decider_map filled, true bo1): 1
Forfeit/unknown (score <=2, no map info at all): 44


## 5. Other data quality issues

In [ ]:
print("event_type unique values:", df['event_type'].unique())
print("match_number null count:", df['match_number'].isna().sum(), "/", len(df))

event_type unique values: ['LAN' 'Online' 'online' 'lan']
match_number null count: 7033 / 7033


## Summary of findings (feeds into 02_preprocessing.ipynb)

- ~76 columns confirmed STATIC (excluded from modeling)
- 965 rows corrected to bo1 (Case A: round-scale score >2, score also normalized to 1-0/0-1)
- 1 row corrected to bo1 (Case C: score ≤2, but decider_map filled)
- 44 rows dropped as forfeit/unknown (score ≤2, no map info at all)
- event_type needs lowercase normalization (4 → 2 values)
- match_number is 100% empty, will be dropped
- team2_player_5/4_* and team1_player_5/4_* show elevated null rates (~13%/~3%), consistent with 4-player rosters; no modeling impact since group is STATIC